<div style="background:#1a1a2e;color:#eee;padding:24px 32px;border-radius:10px;font-family:monospace">
<div style="font-size:11px;color:#aaa;letter-spacing:2px;text-transform:uppercase">Senior GenAI Engineer Programme · Section 1 · Week 1 · Checkpoint</div>
<div style="font-size:24px;font-weight:700;margin:8px 0 4px">Week 1 Checkpoint — <code>jsonl-filter</code> CLI</div>
<div style="font-size:13px;color:#ccc">Tier T0 — Platform Engineer &nbsp;|&nbsp; Anthrolytics &nbsp;|&nbsp; ~3 hours independent work</div>
<div style="font-size:12px;color:#888;margin-top:8px">Prerequisites: Modules 01_01_01 · 01_01_02 · 01_01_03</div>
</div>

## What this notebook is

This is a **project brief**, not a teaching notebook. There are no worked examples, no consolidation pauses, no step-by-step explanations. You have completed three modules this week. This checkpoint asks you to synthesise them independently.

What is provided:
- Priya's requirements (functional and non-functional)
- The grading rubric Marcus will apply
- A starter scaffold (directory structure + empty files)
- One reused component: the `read_jsonl` implementation from Module 1.2
- A `pyproject.toml` stub
- One test stub showing the fixture pattern

What is not provided: how to implement it. That is the point.

**Submit your PR before 3 PM Friday.** Marcus reviews it. Priya makes the call on week 1 completion.

---
## The brief

**Slack — Priya Mehta (CTO) → you, Friday 8:45 AM**

> "End of week 1. Here's your checkpoint. We have a daily job that needs to filter contract metadata — take the raw JSONL dump from S3, keep only the contracts matching a predicate, write the matches to stdout. This is replacing a pandas script that OOM'd on last month's dump. Your version must use a generator pipeline, never load more than one record into memory at a time, handle keyboard interrupt without a traceback, and ship as an installable CLI tool so Ravi can run it from any environment. I need it by 3 PM. Marcus will review the PR."

---

**What Priya is replacing:** A pandas script that does `df = pd.read_json(path, lines=True)` — loads the entire JSONL file into a DataFrame. Last month's dump was 4 GB. It OOM'd. Your version must work on any size file with constant memory.

**What success looks like:** Ravi runs `uv pip install -e .` in a fresh virtualenv, then `jsonl-filter --input contracts.jsonl --field status --value pending` and gets matching records on stdout with no tracebacks. Memory usage stays flat at ~1 record at a time regardless of file size.

---
## Functional requirements

The `jsonl-filter` CLI must support three filter modes:

**1. Field equality:**
```bash
jsonl-filter --input contracts.jsonl --field status --value pending
# Keeps records where record["status"] == "pending"
```

**2. Field contains:**
```bash
jsonl-filter --input contracts.jsonl --field firm --contains "Brennan"
# Keeps records where "Brennan" in record["firm"]
```

**3. Expression filter:**
```bash
jsonl-filter --input contracts.jsonl --expr "status == 'pending' and firm == 'Brennan & Carson'"
# Evaluates a simple field-comparison expression against each record
```

**Output behaviour:**
- Matching records → stdout (one JSON object per line)
- Progress (records read, records matched, elapsed time) → stderr
- Clean on `KeyboardInterrupt`: print summary to stderr, exit with code 130, NO traceback
- Malformed JSON line: log warning to stderr, skip, continue
- Missing field in filter: log warning to stderr, skip record, continue
- File not found: clear error message to stderr, exit with code 1

---

## Non-functional requirements

- **Generator pipeline** — `read_jsonl()` → filter generator → stdout write. No `list()` materialisation anywhere in the pipeline.
- **Peak memory** — must not exceed 2× the size of a single record, regardless of file size.
- **Installable** — `uv pip install -e .` followed by `jsonl-filter --help` must work from a fresh virtualenv.
- **`pyproject.toml`** — with `[project.scripts]` entry point. No `setup.py`.
- **`mypy --strict`** — passes on the package source.
- **`ruff check`** — passes.
- **At least one unit test** — verifies the pipeline is lazy (never materialises more than one record at a time).

---
## Grading rubric — Marcus's review criteria

Scored 1–5. **Passing: average ≥ 4.0, no dimension below 3.**

| Dimension | What Marcus looks for | Weight |
|---|---|---|
| **Generator discipline** | Pipeline is truly lazy — `read_jsonl` + filter generator + stdout write, never a `list()` materialisation anywhere | 1.5× |
| **Keyboard interrupt handling** | `KeyboardInterrupt` caught at top level of `main()`, summary printed to stderr, exit code 130, no traceback | 1.0× |
| **Error handling** | Malformed JSON: warn + skip. Missing field: warn + skip. File not found: clear message + exit code 1. All via `logging`, not `print`. | 1.0× |
| **CLI design** | `--help` is useful. Arguments validated. All three filter modes work. Progress on stderr, data on stdout. | 1.0× |
| **Packaging** | `pyproject.toml` correct. `uv pip install -e .` works. `jsonl-filter` console script works from a fresh venv. No `setup.py`. | 1.5× |
| **Type safety** | `mypy --strict` passes. Types are meaningful — no `Any` everywhere. | 1.0× |
| **Test coverage** | Lazy evaluation test. At least one test per filter mode. At least one error-path test. | 1.0× |
| **Code clarity** | A new engineer reads and understands in 5 minutes. Functions are named. No clever tricks without comments. | 0.5× |

---

**Marcus's comments if you fail generator discipline:**
> "Line NN: `results = list(filter_gen(read_jsonl(path)))` — this loads the entire file into memory. That's the thing we were replacing. The whole point of the generator pipeline is that `read_jsonl` yields one record at a time, the filter generator yields one record at a time, and the write loop handles one record at a time. No `list()` anywhere except tests."

**If `KeyboardInterrupt` handling is missing:**
> "Try running this against a large file and pressing Ctrl+C. You get a traceback. Users don't want a traceback — they want a clean summary. Catch `KeyboardInterrupt` at the top level of `main()`."

**If packaging is broken:**
> "I ran `uv pip install -e .` in a fresh venv and `jsonl-filter: command not found`. Check your `[project.scripts]` in `pyproject.toml`. The entry point must be `jsonl-filter = \"anthrolytics.tools.jsonl_filter.cli:main\"`."

**If everything passes:**
> "Clean. Generator pipeline is correct — I tested it with the 2 GB dump and memory stayed flat. Packaging works. Types are correct. One thing for future: add a `--limit N` flag so we can test quickly on large files. Not blocking — add it next sprint. Approved."

---
## Starter scaffold

Create this directory structure in your repo:

```
tools/
  jsonl_filter/
    __init__.py          ← empty
    reader.py            ← import read_jsonl from Module 1.2 here
    filters.py           ← write your filter generators here
    cli.py               ← write your CLI entry point here
  tests/
    test_jsonl_filter.py ← write your tests here
  pyproject.toml         ← stub provided below (fill in [project.scripts])
```

### Provided: `read_jsonl` from Module 1.2

You do not need to re-implement this. Import it or copy it into `reader.py`:

```python
# tools/jsonl_filter/reader.py
import json
import sys
from typing import Generator

def read_jsonl(path: str) -> Generator[dict, None, None]:
    """
    Generator: yields one parsed dict per line from a JSONL file.
    Skips empty lines silently.
    Warns on JSONDecodeError and continues — does not raise.
    """
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Warning: {path}: malformed JSON — {e}", file=sys.stderr)
```

### Provided: `pyproject.toml` stub

```toml
[project]
name = "anthrolytics-tools"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "typer>=0.9",
]

# TODO: add [project.scripts] entry point here
# jsonl-filter = "???"

[tool.ruff]
line-length = 88
select = ["E", "F", "I"]

[tool.mypy]
strict = true
```

Fill in the `[project.scripts]` section. The entry point is `cli:main` — the `main()` function in `cli.py`.

---
## Provided: test stub

```python
# tools/tests/test_jsonl_filter.py
import io
import json
import pytest
from tools.jsonl_filter.filters import filter_by_field
from tools.jsonl_filter.reader import read_jsonl


SAMPLE_JSONL = "\n".join([
    json.dumps({"contract_id": "NDA-001", "firm": "Brennan & Carson", "status": "pending"}),
    json.dumps({"contract_id": "MSA-002", "firm": "Google LLC",       "status": "processed"}),
    json.dumps({"contract_id": "SOW-003", "firm": "Brennan & Carson", "status": "pending"}),
])


def test_pipeline_is_lazy(tmp_path):
    """The pipeline must never have more than 1 record in memory at a time."""
    # Write sample data to a temp file
    p = tmp_path / "contracts.jsonl"
    p.write_text(SAMPLE_JSONL)

    records_yielded = []
    records_in_memory = [0]   # mutable counter

    original_loads = json.loads

    def counting_loads(s):
        record = original_loads(s)
        records_in_memory[0] += 1
        return record

    # TODO: patch json.loads with counting_loads, iterate the pipeline,
    # and assert records_in_memory[0] never exceeds 1 at any point in the loop.
    # (Hint: check after each yield, before the next json.loads call)
    pass


def test_filter_by_field_equality(tmp_path):
    """filter_by_field yields only records matching field == value."""
    # TODO: implement
    pass


def test_filter_by_field_missing_field(tmp_path, capfd):
    """Records missing the filter field are skipped with a warning."""
    # TODO: implement
    pass
```

---
## Implementation notes

These are hints, not instructions. You have everything you need from the three modules.

### filters.py — the generator pipeline

Each filter mode is a generator that wraps `read_jsonl`. Think of it as a stage in the pipeline:

```python
# The pipeline structure (all generators)
records = read_jsonl(path)                  # source: one record at a time
filtered = filter_by_field(records, ...)    # stage: yields matching records
# write filtered to stdout — one record at a time
```

Each filter generator receives an iterable of records and yields a subset. No list. No materialisation.

### cli.py — the entry point

`typer` is the recommended CLI library. The `main()` function is the entry point. Keyboard interrupt is caught at the top level:

```python
import typer
import sys

app = typer.Typer()

@app.command()
def main(...):
    try:
        # ... generator pipeline and write loop
    except KeyboardInterrupt:
        # print summary to stderr
        raise SystemExit(130)
```

### Expression filter — safe eval

The `--expr` mode evaluates a simple string expression against each record. Do not use `eval()` on arbitrary input — restrict to field comparisons. One approach: parse the expression as `field op value` using `re` and evaluate it as a comparison, not as arbitrary Python.

An alternative (simpler, acceptable for this checkpoint): use `eval()` with a restricted namespace: `eval(expr, {"__builtins__": {}}, record)` — this gives the expression access only to the record's fields. It is not production-safe (an attacker can escape the sandbox) but it is acceptable for this checkpoint's scope.

---
## Self-check before submitting

Run through this before opening the PR. Marcus will check every item.

```python
# Run these in your terminal before submitting:
#
# 1. Install in a fresh venv
#    uv venv /tmp/test_venv && source /tmp/test_venv/bin/activate
#    uv pip install -e .
#    jsonl-filter --help                           # must work
#
# 2. Test all three filter modes
#    echo '{"contract_id":"NDA-001","status":"pending"}' > /tmp/test.jsonl
#    echo '{"contract_id":"MSA-002","status":"processed"}' >> /tmp/test.jsonl
#    jsonl-filter --input /tmp/test.jsonl --field status --value pending
#    # Expected: {"contract_id": "NDA-001", "status": "pending"}
#
# 3. Type check
#    mypy --strict tools/jsonl_filter/
#
# 4. Lint
#    ruff check tools/jsonl_filter/
#
# 5. Tests
#    pytest tools/tests/ -v
#
# 6. KeyboardInterrupt
#    jsonl-filter --input /dev/urandom --field x --value y
#    # Press Ctrl+C — must see summary on stderr, NO traceback
```

In [ ]:
# Local correctness check — run this cell after you have implemented the package
# (This is not Marcus's full review — just a quick sanity check)

import json
import io
import sys

# Minimal pipeline smoke test using only stdlib
sample = [
    '{"contract_id": "NDA-001", "firm": "Brennan & Carson", "status": "pending"}',
    '{"contract_id": "MSA-002", "firm": "Google LLC", "status": "processed"}',
    '{"contract_id": "SOW-003", "firm": "Brennan & Carson", "status": "pending"}',
]

def read_jsonl_inline(lines):
    """Inline read_jsonl for smoke test."""
    for line in lines:
        line = line.strip()
        if line:
            yield json.loads(line)

def filter_by_field(records, field, value):
    """Inline filter for smoke test."""
    for record in records:
        if record.get(field) == value:
            yield record

# Full pipeline — all generators, no list()
pipeline = filter_by_field(read_jsonl_inline(sample), field="status", value="pending")

print("Pipeline output (pending contracts only):")
for record in pipeline:
    print(json.dumps(record))

print()
print("If your implementation is correct, running the above with your actual")
print("filter_by_field from tools/jsonl_filter/filters.py should produce the same output.")
print()
print("Next step: open a terminal and run the self-check commands above.")

---
## Bridge to Week 2

The `jsonl-filter` CLI you built this week is the spine of the programme. It does not stop here.

**Week 2 (Modules 2.1–2.3 + Checkpoint):** You extend `jsonl-filter` with Pydantic validation. Input records are validated against a `ContractRecord` schema before filtering. Invalid records are rejected with structured error messages, not just logged. The `pyproject.toml` gains a new dependency. The same CLI, better-typed.

**Week 3:** The CLI gains an LLM call — it sends matched contracts to the Anthropic API for clause extraction. The rate limiter and retry wrapper from Module 1.3 are what wrap that call.

**Week 4 Capstone:** The CLI becomes `llm-cli` — the full production tool with logging, profiling, packaging, and a test suite. The `jsonl-filter` you ship today is the first version of the thing Ravi will run in production.

Every line you write today will be in production by week 4.